# 03 – Fra piksler til prediksjoner

**Tid:** 75 minutter. **Mål:** følge formene fra bilder til klasseskårer,
forstå vektede summer og bygge en lineær modell. Modellen trenes ikke ennå.

Vi starter med vektede summer og bygger deretter en modell med ett lineært lag.
Vi bygger først ett lineært lag; det skjulte laget legger vi til på dag 2.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Eksempel: en vektet sum

En lineær modell multipliserer hvert inndatatall med en vekt, summerer og
legger til et konstantledd (*bias*). Vektene bestemmer påvirkningen fra hver verdi.
I uttrykket `x @ W + b` er `@` matrisemultiplikasjon, ikke elementvis gange.


In [ ]:
# To målinger gir én skår. Tallene er valgt for hånd.
x = torch.tensor([[2.0, 3.0]])
weights = torch.tensor([[0.5], [-1.0]])
bias = torch.tensor([0.2])
score = x @ weights + bias
print("Skår:", score, "Form:", score.shape)
print("Samme beregning for hånd:", 2 * 0.5 + 3 * (-1) + 0.2)


## Oppgave 1: Endre én vekt og forutsi virkningen

Gjenbruk tallene fra tensoroppgave 3 i denne nye kjernen:
`inputs = [[1., 2., 3.], [4., 5., 6.]]`,
`weights = [[1., 0.], [0., 1.], [1., -1.]]`, `bias = [0.5, -0.5]`.
Lag tensorene og beregn `scores`. Øk så vekten i rad 0, kolonne 0 med 1
og beregn `changed_scores`. Hvilke utganger endres, og med hvor mye?
Skriv forventningen i svarcellen før kjøring.

<details><summary>Hint</summary>

Kopier vektene med `clone()`. Vektens rad velger inngang, kolonnen velger utgang.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert torch.allclose(scores, torch.tensor([[4.5, -1.5], [10.5, -1.5]]))

assert torch.allclose(changed_scores - scores, torch.tensor([[1., 0.], [4., 0.]]))


### Refleksjon

Hva representerer radene og kolonnene i resultatet? Hvorfor må de to indre
dimensjonene stemme ved matrisemultiplikasjon? Hvordan ville `*` oppført seg her?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.

Hvilken klassekolonne endret seg? Knytt størrelsen på endringen til inngangsverdiene.


**Mitt svar:**

Skriv din forklaring her.


## Eksempel: flat ut bilder, behold bildeskillet

En lineær modell trenger én rekke tall per bilde. `flatten(start_dim=1)`
beholder dimensjon 0 og slår sammen resten. Piksler fjernes ikke, men den
todimensjonale plasseringen er ikke lenger eksplisitt i formen.


In [ ]:
small_images = torch.arange(24, dtype=torch.float32).reshape(2, 1, 3, 4)
flattened = small_images.flatten(start_dim=1)
print(small_images.shape, "→", flattened.shape)


## Nå trenger vi grupper av bilder

Til nå har vi hentet enkeltbilder med `train_data[index]`. Modellen
kan regne på flere bilder samtidig, og under trening vil vi oppdatere
den etter hver lille gruppe. En slik gruppe kalles en **batch**.

`DataLoader` samler bilder og etiketter fra datasettet. Oppsettet er
synlig under: `batch_size=64` gir opptil 64 bilder, og `shuffle=True`
stokker rekkefølgen for hver gjennomgang. Den siste gruppen kan være mindre.
Datasettet inneholder eksemplene; lasteren bestemmer hvordan de grupperes.
`next(iter(...))` henter én gruppe som vi kan undersøke.


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
images, labels = next(iter(train_loader))
print("Bilder:", images.shape, images.dtype)
print("Etiketter:", labels.shape, labels.dtype)


### Refleksjon

Forklar hvert tall i bildegruppens form. Hvorfor har etikettene bare
én dimensjon? Hva ville endret seg om `batch_size` var 16?

Skriv svaret ditt i Markdown-cellen under.


**Mitt svar:**

Skriv din forklaring her.


## Oppgave 2: Lag ti skårer for MNIST

Hent `images, labels` fra `train_loader`. Lag `flat_images` med én rad
per bilde. Lag tilfeldige `mnist_weights` med formen `[784, 10]`, gang dem med
0.01, og lag `mnist_bias` som ti nuller. Beregn `mnist_scores`.
Bruk `argmax(dim=1)` til å lage `predictions`. Skriv ut alle formene.

<details>
<summary>Hint</summary>

`torch.randn` lager tilfeldige tall. `argmax(dim=1)` finner kolonnen med høyest skår i hver rad.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert flat_images.shape == (images.shape[0], 784)
assert mnist_scores.shape == (images.shape[0], 10)
assert predictions.shape == labels.shape


## Eksempel: la `nn.Linear` håndtere parametrene

`nn.Linear(3, 2)` lærer tre vekter og ett konstantledd per utgang.
Den lagrer vektene som `[utganger, inputs]`, motsatt av vår håndlagde
matrise. Laget utfører den nødvendige transponeringen selv.
`nn.Sequential` kjører flere lag etter hverandre.

Med vår matrisenotasjon er dette `X @ layer.weight.T + layer.bias`.
Antall parametere er `inputs * utganger + utganger`.


In [ ]:
small_layer = nn.Linear(3, 2)
print("Resultat:", small_layer(torch.ones(4, 3)).shape)
print("Vekter:", small_layer.weight.shape, "Konstantledd:", small_layer.bias.shape)


## Oppgave 3: Bygg en lineær bildemodell

Lag `model = nn.Sequential(...)` med `nn.Flatten()` og et passende `nn.Linear`.
Modellen skal motta bildene direkte og gi ti skårer per bilde.
Beregn `model_scores`, og tell parametrene med
`sum(parameter.numel() for parameter in model.parameters())`.

<details>
<summary>Hint</summary>

Hvert bilde har `28 * 28 = 784` piksler. Det er ti mulige sifre.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert model_scores.shape == (len(images), 10)
assert sum(p.numel() for p in model.parameters()) == 7850


### Refleksjon

Skriv formene gjennom modellen for en gruppe på 32 MNIST-bilder.
Hvorfor har modellen 7 850 parametere? Hvorfor forventer vi ikke gode
prediksjoner før trening? Er de ti skårene sannsynligheter?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Skriv din forklaring her.


## Valgfri oppgave 4: Er matrisemodellen og `nn.Linear` samme beregning?

Lag `equivalent_layer = nn.Linear(3, 2)`. Under `torch.no_grad()`, kopier `weights.T`
til `equivalent_layer.weight` og `bias` til `equivalent_layer.bias` med `copy_`.
Beregn `layer_output = equivalent_layer(inputs)` og sammenlign med `scores`.


In [ ]:
# Skriv din kode her.


In [ ]:
assert torch.allclose(layer_output, scores)


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Start her:** [PyTorch – Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html). Følg spesielt `Flatten`, `Linear` og `Sequential`, og sammenlign tensorformene med vår modell. Egen modellklasse og valg av GPU kan du hoppe over nå.
- **Visuell forklaring:** [3Blue1Brown – But what is a Neural Network?](https://www.3blue1brown.com/lessons/neural-networks/). Video og illustrert tekst med håndskrevne sifre som eksempel. Se etter koblingen mellom piksler, vekter og lag; aktiveringsfunksjonene kan avvike fra ReLU i kurset vårt.
